# 06 — Финальная оценка моделей

Этот ноутбук объединяет результаты зафиксированных финальных тестовых запусков и подводит итог исследованию автоматизации feature engineering для задачи ранжирования музыкальных рекомендаций.

В отличие от предыдущих ноутбуков, здесь модели повторно не обучаются. Используются значения из сохранённого финального запуска, а результаты представляются в виде сравнительных таблиц и статистических оценок.

## Что рассматривается

**1. Основная ranking-постановка**

Сравниваются случайное ранжирование, популярность, ручные наборы признаков, автоматически отобранные наборы RFE и полный расширенный пул `POOL2` из 97 признаков.

**2. Реалистичное множество кандидатов**

Проверяется качество при наличии непрослушанных отрицательных кандидатов. Сравниваются ручной набор `manual18`, расширенный пул `pool90`, модели с collaborative filtering (CF), компактные RFE-модели и вариант объединения предсказаний.

**3. Масштабирование collaborative filtering**

Исследуется, как размер графа взаимодействий влияет на общую NDCG@10 и качество ранжирования новых user-item пар.

Для сравнения используются NDCG@10, Recall@10 и bootstrap-доверительные интервалы различий между моделями.

## 1. Импорты и служебные функции

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)


def find_project_file(relative_path: str):
    """Ищет файл как из корня репозитория, так и при запуске из notebooks/."""
    candidates = [
        Path(relative_path),
        Path("..") / relative_path,
        Path("/kaggle/working") / relative_path,
    ]
    for p in candidates:
        if p.exists():
            return p
    return None


log_path = find_project_file("results/cf_and_final_test.txt")
if log_path is not None:
    print("Финальный лог найден:", log_path)
else:
    print("Финальный лог не найден локально. Таблицы ниже содержат значения из зафиксированного финального запуска.")

## 2. Финальный test на основной ranking-постановке

На этом этапе сравниваются базовые и автоматически отобранные наборы признаков. Значения ниже взяты из финального locked test-run.

- `M1` — базовый ручной набор из 26 признаков;
- `M2` — более сильный набор из 48 признаков;
- `RFE-26` и `RFE-48` — автоматически отобранные подмножества;
- `POOL2` — полный расширенный пул из 97 признаков.

In [ ]:
main_test = pd.DataFrame([
    {"model": "random",                         "features": 0,  "test_ndcg10": 0.6926, "std_seed": np.nan},
    {"model": "popularity (gitem_rate_30d)",   "features": 0,  "test_ndcg10": 0.7138, "std_seed": np.nan},
    {"model": "M1",                             "features": 26, "test_ndcg10": 0.7721, "std_seed": 0.0005},
    {"model": "M2",                             "features": 48, "test_ndcg10": 0.7812, "std_seed": 0.0002},
    {"model": "RFE-26",                         "features": 26, "test_ndcg10": 0.7824, "std_seed": 0.0002},
    {"model": "RFE-48",                         "features": 48, "test_ndcg10": 0.7830, "std_seed": 0.0003},
    {"model": "POOL2",                          "features": 97, "test_ndcg10": 0.7839, "std_seed": 0.0004},
])

main_test.sort_values("test_ndcg10", ascending=False).reset_index(drop=True)

### Статистические сравнения

In [ ]:
main_ci = pd.DataFrame([
    {"comparison": "M2 - M1",        "delta": 0.0091, "ci_low": 0.0073, "ci_high": 0.0110},
    {"comparison": "RFE-26 - M1",    "delta": 0.0104, "ci_low": 0.0083, "ci_high": 0.0124},
    {"comparison": "RFE-26 - M2",    "delta": 0.0013, "ci_low": -0.0001, "ci_high": 0.0025},
    {"comparison": "RFE-48 - M2",    "delta": 0.0019, "ci_low": 0.0007, "ci_high": 0.0030},
    {"comparison": "POOL2 - M2",     "delta": 0.0028, "ci_low": 0.0017, "ci_high": 0.0038},
    {"comparison": "POOL2 - RFE-48", "delta": 0.0009, "ci_low": 0.0002, "ci_high": 0.0017},
    {"comparison": "POOL2 - M1",     "delta": 0.0119, "ci_low": 0.0098, "ci_high": 0.0141},
])

main_ci

**Вывод по основной постановке.** Лучшее абсолютное качество показывает полный `POOL2` (`NDCG@10 = 0.7839`). При этом `RFE-48` достигает `0.7830`, то есть почти сохраняет качество полного 97-признакового пула при вдвое меньшем числе признаков. `RFE-26` также превосходит ручной `M1` и практически не уступает `M2`. Это подтверждает основной вывод работы: автоматический отбор позволяет получить компактный сильный набор без ручного выбора признаков.

## 3. Финальный test на реалистичном candidate set

Во второй постановке к положительным взаимодействиям добавляются непрослушанные треки. Такой тест сложнее и ближе к реальному ранжированию, потому что модель должна отличать релевантные объекты от дополнительных отрицательных кандидатов.

Отдельно оцениваются:

- общий `NDCG@10`;
- `NDCG@10` только для новых user-item пар;
- `Recall@10`;
- recall для новых и повторных пар.

In [ ]:
realistic_test = pd.DataFrame([
    {"model": "manual18",                       "split": "rep",  "ndcg10": 0.756936, "recall10": 0.359020, "recall_new": 0.115340, "recall_repeat": 0.454174, "ndcg10_new": 0.638752},
    {"model": "manual18",                       "split": "TEST", "ndcg10": 0.749720, "recall10": 0.352483, "recall_new": 0.118801, "recall_repeat": 0.443415, "ndcg10_new": 0.635020},
    {"model": "pool90",                         "split": "rep",  "ndcg10": 0.780751, "recall10": 0.376978, "recall_new": 0.110363, "recall_repeat": 0.487581, "ndcg10_new": 0.752680},
    {"model": "pool90",                         "split": "TEST", "ndcg10": 0.776213, "recall10": 0.371749, "recall_new": 0.110094, "recall_repeat": 0.481022, "ndcg10_new": 0.744544},
    {"model": "pool90+CF",                      "split": "rep",  "ndcg10": 0.782548, "recall10": 0.378760, "recall_new": 0.116480, "recall_repeat": 0.484586, "ndcg10_new": 0.768946},
    {"model": "pool90+CF",                      "split": "TEST", "ndcg10": 0.777746, "recall10": 0.372993, "recall_new": 0.117836, "recall_repeat": 0.476906, "ndcg10_new": 0.762395},
    {"model": "CF + popularity",                "split": "rep",  "ndcg10": 0.736972, "recall10": 0.354040, "recall_new": 0.198643, "recall_repeat": 0.407653, "ndcg10_new": 0.745700},
    {"model": "CF + popularity",                "split": "TEST", "ndcg10": 0.729296, "recall10": 0.348805, "recall_new": 0.211057, "recall_repeat": 0.396026, "ndcg10_new": 0.744589},
    {"model": "RFE-18 from pool90",             "split": "rep",  "ndcg10": 0.776408, "recall10": 0.375066, "recall_new": 0.106818, "recall_repeat": 0.487413, "ndcg10_new": 0.742564},
    {"model": "RFE-18 from pool90",             "split": "TEST", "ndcg10": 0.772069, "recall10": 0.369847, "recall_new": 0.108613, "recall_repeat": 0.479735, "ndcg10_new": 0.735384},
    {"model": "RFE-18 from pool90+CF",          "split": "rep",  "ndcg10": 0.775313, "recall10": 0.375641, "recall_new": 0.112315, "recall_repeat": 0.484381, "ndcg10_new": 0.764446},
    {"model": "RFE-18 from pool90+CF",          "split": "TEST", "ndcg10": 0.770797, "recall10": 0.370506, "recall_new": 0.115306, "recall_repeat": 0.476253, "ndcg10_new": 0.757796},
    {"model": "blend pool90 + CF model (w=0.05)","split": "rep",  "ndcg10": 0.780874, "recall10": 0.377508, "recall_new": 0.110579, "recall_repeat": 0.487808, "ndcg10_new": 0.754787},
    {"model": "blend pool90 + CF model (w=0.05)","split": "TEST", "ndcg10": 0.776572, "recall10": 0.371994, "recall_new": 0.110965, "recall_repeat": 0.480726, "ndcg10_new": 0.747064},
])

test_only = realistic_test[realistic_test["split"] == "TEST"].copy()
test_only.sort_values("ndcg10", ascending=False).reset_index(drop=True)

### Ключевые bootstrap-сравнения на TEST

In [ ]:
realistic_ci = pd.DataFrame([
    {"comparison": "pool90 - manual18",                 "metric": "ndcg",     "delta": 0.0265, "ci_low": 0.0238, "ci_high": 0.0296},
    {"comparison": "pool90 - manual18",                 "metric": "ndcg_new", "delta": 0.1095, "ci_low": 0.1023, "ci_high": 0.1169},
    {"comparison": "pool90+CF - pool90",                "metric": "ndcg",     "delta": 0.0015, "ci_low": 0.0007, "ci_high": 0.0024},
    {"comparison": "pool90+CF - pool90",                "metric": "ndcg_new", "delta": 0.0179, "ci_low": 0.0154, "ci_high": 0.0202},
    {"comparison": "pool90+CF - manual18",              "metric": "ndcg",     "delta": 0.0280, "ci_low": 0.0252, "ci_high": 0.0311},
    {"comparison": "pool90+CF - manual18",              "metric": "ndcg_new", "delta": 0.1274, "ci_low": 0.1200, "ci_high": 0.1348},
    {"comparison": "RFE18(pool90+CF) - RFE18(pool90)",  "metric": "ndcg",     "delta": -0.0013,"ci_low": -0.0028,"ci_high": 0.0004},
    {"comparison": "RFE18(pool90+CF) - RFE18(pool90)",  "metric": "ndcg_new", "delta": 0.0224, "ci_low": 0.0197, "ci_high": 0.0251},
])

realistic_ci

**Вывод по realistic candidates.** `pool90` существенно превосходит `manual18`. Добавление collaborative filtering к полному пулу даёт небольшой, но статистически подтверждённый общий прирост (`+0.0015 NDCG@10` на TEST), а для новых user-item пар эффект намного сильнее (`+0.0179`). Это показывает, что CF наиболее полезен именно там, где прямой истории конкретной пары ещё нет.

## 4. Масштабирование collaborative filtering

In [ ]:
cf_scale = pd.DataFrame([
    {"corpus": "no CF", "users_in_graph": 0,    "model": "pool90",           "ndcg10": 0.780751, "ndcg10_new": 0.752680, "recall_new": 0.110363},
    {"corpus": "1371",  "users_in_graph": 1371, "model": "pool90+CF",        "ndcg10": 0.781958, "ndcg10_new": 0.764219, "recall_new": 0.114251},
    {"corpus": "3000",  "users_in_graph": 3000, "model": "pool90+CF",        "ndcg10": 0.781422, "ndcg10_new": 0.765359, "recall_new": 0.114217},
    {"corpus": "6000",  "users_in_graph": 6000, "model": "pool90+CF",        "ndcg10": 0.781998, "ndcg10_new": 0.768701, "recall_new": 0.117220},
    {"corpus": "all",   "users_in_graph": 9238, "model": "pool90+CF",        "ndcg10": 0.782548, "ndcg10_new": 0.768946, "recall_new": 0.116480},
])

cf_scale

Масштабирование CF-корпуса не приводит к резкому росту общей NDCG, но эффект на новых парах остаётся устойчивым и в целом усиливается при использовании большего графа взаимодействий.

## 5. Итоговая сводка

In [ ]:
summary = pd.DataFrame([
    {
        "result": "Best main-test model",
        "value": "POOL2: NDCG@10 = 0.7839",
        "interpretation": "Лучшее абсолютное качество на основной постановке"
    },
    {
        "result": "Compact model",
        "value": "RFE-48: NDCG@10 = 0.7830",
        "interpretation": "Почти качество POOL2 при 48 вместо 97 признаков"
    },
    {
        "result": "Realistic candidate set",
        "value": "pool90+CF: NDCG@10 = 0.7777",
        "interpretation": "Лучший общий результат среди сравниваемых финальных вариантов"
    },
    {
        "result": "CF effect on new pairs",
        "value": "+0.0179 NDCG@10",
        "interpretation": "Главный дополнительный выигрыш CF проявляется на новых user-item парах"
    },
])

summary

## 6. Финальный вывод

Итоговые тесты подтверждают основные выводы исследования:

1. **Расширенный feature pool устойчиво сильнее небольшого ручного набора.**
2. **RFE позволяет автоматически получить компактный набор признаков почти без потери качества.**
3. **Полный отказ от агрегированных признаков в пользу более сложных представлений не оказался выгоднее сильного boosting baseline.**
4. **Collaborative filtering даёт дополнительный сигнал прежде всего для новых user-item пар.**
5. Наиболее практичная стратегия для этой задачи — широкий автоматически построенный набор временных агрегатов + автоматический feature selection + специализированные CF-сигналы.

Таким образом, основной результат работы — не замена feature engineering одной сложной моделью, а **систематизация и автоматизация генерации и отбора признаков при сохранении сильного качества ранжирования**.

## 7. Сохранение компактных итоговых таблиц

Ячейка ниже необязательна. Она сохраняет только небольшие CSV-файлы с финальными результатами и **не запускает обучение**.

In [ ]:
out_dir = Path("results")
if not out_dir.exists() and Path("..").exists():
    out_dir = Path("..") / "results"
out_dir.mkdir(parents=True, exist_ok=True)

main_test.to_csv(out_dir / "final_main_test.csv", index=False)
main_ci.to_csv(out_dir / "final_main_test_ci.csv", index=False)
realistic_test.to_csv(out_dir / "final_realistic_candidates.csv", index=False)
realistic_ci.to_csv(out_dir / "final_realistic_ci.csv", index=False)
cf_scale.to_csv(out_dir / "final_cf_scaling.csv", index=False)

print("Итоговые таблицы сохранены в:", out_dir.resolve())